# Test du Pipeline de Vision - Pose 2D et 3D

Ce notebook teste le pipeline de détection de pose sur une image locale.

- Encapsuler les classes pour pouvoir changer les modèles facilement
- Finir d'implémenter le test avec mediapipe
- Rappeler Angelo pour le push avec les compatibilités mediapipe

## 1. Import des Bibliothèques Requises

Importe les modules nécessaires pour MediaPipe, OpenCV, Matplotlib et le pipeline de pose.

In [ ]:
from humanoid_mimic_pose.vision.pose_detector import PoseDetector
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from humanoid_mimic_pose.vision.camera import Camera
import mediapipe as mp


## Charger une Image depuis le Disque Local

Ouvre un dialogue de sélection de fichier pour choisir directement une image depuis votre PC.

In [ ]:
# Sélectionner une image via un dialogue de fichier
from tkinter import filedialog
import tkinter as tk

# Créer une fenêtre Tkinter invisible et la placer au premier plan
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
root.lift()
root.update()

# Ouvrir le dialogue de sélection de fichier
image_path = filedialog.askopenfilename(
    parent=root,
    title="Sélectionnez une image",
    filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp *.gif"), ("Tous les fichiers", "*.*")]
)

root.destroy()

# Charger l'image
if image_path and Path(image_path).exists():
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        image_rgb = None
        print(f"Impossible de lire l'image : {image_path}")
    else:
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        print(f"Image chargée avec succès : {image_path}")
        print(f"  Dimensions : {image_rgb.shape}")
        
        # Afficher l'image originale
        plt.figure(figsize=(10, 8))
        plt.imshow(image_rgb)
        plt.title("Image Originale")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
else:
    image_rgb = None
    print("Aucune image sélectionnée ou fichier introuvable")

## Utiliser la caméra live

In [ ]:
# récuperer l'image de la caméra

cam = Camera()
frame = cam.read()


pose_detector = PoseDetector()
skeleton = pose_detector.infer(frame)





## 3. Appliquer la Détection de Pose 2D

In [ ]:
import time

In [ ]:
t1 = time.time()
if image_rgb is not None:
    pose_detector = PoseDetector()
    print("En cours : Détection de pose...")
    skeleton = pose_detector.infer(image_rgb)
    t2 = time.time()
    print(f"Pose : Temps d'inférence = {t2 - t1:.2f} secondes")
    if skeleton is None:
        print("Pose : Aucun point détecté - Essayez d'autres images")
else:
    skeleton = None
    print("L'image n'a pas pu être chargée.")

# Afficher les résultats de la détection de pose 2D

In [ ]:
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
def draw_landmarks_on_image(rgb_image, skeleton):
  pose_landmarks_list = skeleton.raw.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=20, circle_radius=10)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

In [ ]:
# annotated_image = draw_landmarks_on_image(frame, skeleton)
annotated_image = draw_landmarks_on_image(image_rgb, skeleton)
print('keypoints 3D : ', skeleton.keypoints3d)
plt.imshow(annotated_image)